In [ ]:
import os
import numpy as np
from design import Design
from geometry import analyze_results, plot_barriers

project_name = "SynRM_test"
design_name = "Design01"
path_data = os.path.join(os.getcwd(), 'data')
path_results = 'results'
for path in [path_data, path_results]:
    os.makedirs(path, exist_ok=True)
file_name_aedt = f'{path_data}/{project_name}.aedt'
plot_design = True
n_designs = 3

# Define constants
AEDT_VERSION = "2024.2"
NUM_CORES = 4
NG_MODE = True  #non-graphical mode
CLS_EXIT = True #close on exit

if not os.path.exists(file_name_aedt):
    design = Design.create(
        project_name, design_name, file_name_aedt,
        version=AEDT_VERSION,
        non_graphical=NG_MODE,
        new_desktop=False,
        close_on_exit=CLS_EXIT,
    )
else:
    design = Design.load(
        file_name_aedt,
        version=AEDT_VERSION,
        non_graphical=NG_MODE,
        new_desktop=False,
        close_on_exit=CLS_EXIT,
    )

In [ ]:
from generators import FourStupid

generator = FourStupid(design)

n_designs = 1
for i in range(0, n_designs):
    # Add rotor
    design.add_rotor()

    # Add barriers
    barriers = generator.random_barriers()
    for barrier in barriers:
        design.add_rotor_barrier(barrier)

    # Compute the torque
    Tor = design.compute(NUM_CORES)
    TorAvg, _, TorRippleRms = analyze_results(Tor)

    # Delete the rotor
    design.delete_rotor()

    # Potentially save the design
    if plot_design:
        title = f'Torque mean value: {np.round(TorAvg,2)} Nm, ripple relative value: {np.round(TorRippleRms,2)} %'
        file_name = f'{path_results}/design_{i}'
        plot_barriers(barriers, design, title=title, file_name = f"{file_name}.png")
        generator.save_barriers(f"{file_name}.npz")

In [ ]:
design.close_project()